# 10 Model Quantization for Edge Deployment
This notebook demonstrates Post-Training Quantization (PTQ) to reduce model size and latency using PyTorch.

In [1]:
import os
import torch
import time
from pneumonia_classifier.ml.model.arch import Net

## Load Best Model

In [2]:
MODEL_PATH = 'best_model_optuna.pt'
model = Net()
model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'))
model.eval()

Net(
  (convolution_block1): Sequential(
    (0): Conv2d(3, 8, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (pooling11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (convolution_block2): Sequential(
    (0): Conv2d(8, 20, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): BatchNorm2d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (pooling22): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (convolution_block3): Sequential(
    (0): Conv2d(20, 10, kernel_size=(1, 1), stride=(1, 1))
    (1): ReLU()
    (2): BatchNorm2d(10, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (pooling33): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (convolution_block4): Sequential(
    (0): Conv2d(10, 20, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2

## Dynamic Quantization (INT8)
Quantizes the weights of linear layers (and optionally others) to INT8.

In [3]:
quantized_model = torch.quantization.quantize_dynamic(
    model, {torch.nn.Linear}, dtype=torch.qint8
)

print("Quantization complete.")

Quantization complete.


C:\Users\user\AppData\Local\Temp\ipykernel_21536\3931604733.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


## Size Comparison

In [4]:
def get_size(path):
    size = os.path.getsize(path)
    return size / (1024 * 1024)

torch.save(model.state_dict(), "fp32_model.pt")
torch.save(quantized_model.state_dict(), "int8_model.pt")

print(f"FP32 Model Size: {get_size('fp32_model.pt'):.2f} MB")
print(f"INT8 Model Size: {get_size('int8_model.pt'):.2f} MB")

FP32 Model Size: 0.06 MB
INT8 Model Size: 0.06 MB


## Latency Benchmark

In [5]:
dummy_input = torch.randn(1, 3, 224, 224)

def benchmark(m, input_data, name):
    start = time.time()
    for _ in range(100):
        m(input_data)
    end = time.time()
    print(f"{name} average latency: {(end - start) / 100 * 1000:.2f} ms")

benchmark(model, dummy_input, "FP32")
benchmark(quantized_model, dummy_input, "INT8 Dynamic")

FP32 average latency: 61.34 ms
INT8 Dynamic average latency: 78.11 ms
